In [173]:
# Importing the xlsx-file and making the data frames.

import sqlite3
import pandas as pd

excel = "C:/Users/omistaja/Desktop/AS Project/Data/Spreadsheets/street_signs_taipei_sheets.xlsx"

sign_df = pd.read_excel(excel, sheet_name="SIGN")
observation_df = pd.read_excel(excel, sheet_name="OBSERVATION")

# Checking column names

print(sign_df.columns.tolist())
print(observation_df.columns.tolist())


['sign_id', 'sign_location', 'sign_notes']
['observation_id', 'sign_id', 'survey_id', 'observation_date', 'observation_time', 'sign_latitude', 'sign_longitude', 'sign_status', 'sign_condition', 'observation_notes']


In [174]:
# Connecting to .sqlite file

conn = sqlite3.connect("taipei_street_signs_sqlite_practice.sqlite")
cur = conn.cursor()
conn.execute("PRAGMA foreign_keys = ON")

In [175]:
# Sheet 1: SIGN

conn.execute("""
CREATE TABLE IF NOT EXISTS SIGN (
    sign_id TEXT PRIMARY KEY,
    sign_location TEXT,
    sign_notes TEXT
    )"""
    )

sign_df.to_sql(
    "SIGN",
    conn,
    if_exists="replace",
    index=False
    )

2

In [176]:
result = pd.read_sql("""
    SELECT sign_id
    FROM SIGN
    """, 
    conn)

result.index = result.index + 1 # The default output makes the first row "0" and it's annoying

result

,sign_id
1,S0001
2,S0002


In [177]:
# Testing SIGN filtering

result_filtered_sign = pd.read_sql("""
    SELECT sign_id
    FROM SIGN
    WHERE sign_location = 'Taiwan'
    """,
    conn  
    )


result_filtered_sign.index = result_filtered_sign.index + 1

result_filtered_sign

,sign_id
1,S0002


In [178]:
# Sheet 1: OBSERVATION

conn.execute("""
CREATE TABLE IF NOT EXISTS OBSERVATION (
    observation_id TEXT PRIMARY KEY,
    sign_id TEXT,
    survey id TEXT,
    observation_date TEXT,
    observation_time TEXT,
    sign_latitude INT,
    sign_longitude INT,
    sign_status TEXT,
    sign_condition TEXT,
    sign_notes TEXT,

    FOREIGN KEY(sign_id) REFERENCES SIGN(sign_id)
    )"""
    )

observation_df.to_sql(
    "OBSERVATION",
    conn,
    if_exists="replace",
    index=False
    )

3

In [179]:
# Testing OBSERVATION filtering

result_filtered_obs = pd.read_sql("""
    SELECT observation_id
    FROM OBSERVATION
    WHERE sign_status = 'Imaginary'
    """,
    conn  
    )


result_filtered_obs.index = result_filtered_obs.index + 1

result_filtered_obs

,observation_id
1,O0001
2,O0003


In [180]:
results_filtered_combo = pd.read_sql("""
    SELECT sign_id, observation_id, sign_status
    FROM OBSERVATION
    WHERE sign_id = (SELECT sign_id FROM SIGN WHERE sign_location = 'Finland')
    """,
    conn,
    )

results_filtered_combo.index = results_filtered_combo.index + 1

results_filtered_combo

,sign_id,observation_id,sign_status
1,S0001,O0001,Imaginary
2,S0001,O0002,Corporeal
